# Ground state search with a vision transformer

This notebook shows how to use the vision transformer (ViT) wave function of Viteritti, Rende and Becca ([Phys. Rev. Lett. 130, 236401 (2023)](https://doi.org/10.1103/PhysRevLett.130.236401)), available as `jVMC_exp.nets.LogViterittiSpatialViTJVMC` (alias `ViterittiViT`), on a two-dimensional lattice. The ViT splits the spin configuration into patches, embeds each patch into a vector, and processes them with attention layers that mix the information of all patches.

As a minimal example we use the 2D transverse-field Ising model on a $2 \times 2$ lattice, small enough to compare with exact diagonalisation.

In [1]:
import numpy as np
import jax.numpy as jnp

import jVMC_exp
import jVMC_exp.nets as nets
import jVMC_exp.operator.discrete as op
import jVMC_exp.sampler as sampler
from jVMC_exp.vqs import NQS

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Running in single-node mode (JVMC_USE_DISTRIBUTED not set)


## Hamiltonian

$$
H = -\sum_{\langle i,j\rangle} \sigma^z_i \sigma^z_j - g \sum_i \sigma^x_i ,
$$

on an $L_x \times L_y$ lattice with periodic boundary conditions, at the critical-region value $g = 1$. A configuration has shape `(Ly, Lx)`, and the operators act on the sites of the flattened configuration, so the site $(x, y)$ has index `y * Lx + x`.

The bonds are collected in a set, so each pair of neighbours is coupled once. On a $2 \times 2$ lattice the bond across the boundary connects the same two sites as the direct one, so there are only 4 distinct bonds.

In [2]:
Lx, Ly = 2, 2
L = Lx * Ly
g = 1.0

def site(x, y):
    return y * Lx + x

bonds = set()
for y in range(Ly):
    for x in range(Lx):
        i = site(x, y)
        bonds.add(tuple(sorted((i, site((x + 1) % Lx, y)))))
        bonds.add(tuple(sorted((i, site(x, (y + 1) % Ly)))))

hamiltonian = 0
for i, j in sorted(bonds):
    hamiltonian += -op.SigmaZ(i) * op.SigmaZ(j)
for i in range(L):
    hamiltonian += -g * op.SigmaX(i)

## Vision transformer

The main hyperparameters of the network are:

- `patch_size`: linear size of the square patches the lattice is split into (here single sites),
- `layers`, `embed_dim`, `heads`: number of transformer layers, dimension of the patch embeddings, and number of attention heads,
- `symmetry_average="d4"`: the wave function is averaged over the 8 symmetries of the square (rotations and reflections), which makes it invariant under them,
- `param_dtype`, `compute_dtype`: data types of the parameters and of the computation; here the parameters are real.

The network returns the logarithm of a positive amplitude, with an optional sign given by `sign_rule` (`"none"`, the default, or `"marshall"` for the Marshall sign rule of the Heisenberg antiferromagnet). A positive wave function is exact for the ground state of the transverse-field Ising model, since all the off-diagonal elements of $H$ are non-positive.

With $2^4 = 16$ basis states, the `ExactSampler` can sum over all of them, so there is no sampling noise.

In [3]:
net = nets.LogViterittiSpatialViTJVMC(
    Lx=Lx,
    Ly=Ly,
    patch_size=1,
    layers=1,
    embed_dim=8,
    heads=2,
    symmetry_average="d4",
    param_dtype=jnp.float32,
    compute_dtype=jnp.float32,
)
psi = NQS(net, (Ly, Lx), batchSize=2**L, seed=1234)
exact_sampler = sampler.ExactSampler(psi)
print("Number of parameters:", psi.numParameters)

Number of parameters: 856


## Optimisation with MinSR

`MinSR` ([Chen and Heyl, arXiv:2302.01941](https://arxiv.org/abs/2302.01941)) computes the same update as stochastic reconfiguration, but solves a linear system whose size is the number of samples instead of the number of parameters. This is much cheaper for large networks like transformers, which have many more parameters than samples (here 856 parameters and 16 configurations). `diagonalShift` regularises the system, and `Pinv` solves it with a pseudo-inverse with relative cutoff `pinv_cutoff`.

In [4]:
loss_function = jVMC_exp.objective_function.Observable(hamiltonian)
stepper = jVMC_exp.stepper.Euler(2e-2)
solver = jVMC_exp.solver.Pinv(pinv_cutoff=1e-8)
opt = jVMC_exp.optimizer.MinSR(exact_sampler, psi, solver=solver, diagonalShift=1e-3)

out = opt.ground_state_search(200, loss_function, stepper)

100%|██████████| 200/200 [00:15<00:00, 12.86it/s, E=-5.2262e+00+0.0000e+00j ± 5.1559e-04 (Var = 1.1540e-06)]

Recorded timings:
    - sampling: 1.974018s
    - compute objective function and gradient: 8.175260s
    - solve: 1.862799s


## Comparison with exact diagonalisation

The `ExactSampler` stores the wave function of the last `sample()` call and does not update it when the parameters change. The last optimisation step updates the parameters after the measurement, so we call `sample()` again before measuring the final energy.

The exact ground state energy is obtained by diagonalising the $16 \times 16$ Hamiltonian matrix.

In [5]:
exact_sampler.sample()
energy = exact_sampler(hamiltonian).mean.real.item()
H_dense = hamiltonian.to_dense(exact_sampler.basis)
exact_energy = np.linalg.eigvalsh(H_dense)[0]

print(f"VMC energy:   {energy:.6f}")
print(f"Exact energy: {exact_energy:.6f}")
print(f"Relative error: {abs((energy - exact_energy) / exact_energy):.1e}")

VMC energy:   -5.226249
Exact energy: -5.226252
Relative error: 6.2e-07
